In [1]:
# CELL 1 — Upload CSV File
from google.colab import files
uploaded = files.upload()
# cleaned_combined.csv select karo

Saving cleaned_combined.csv to cleaned_combined.csv


In [2]:
# CELL 2 — Load Data
import pandas as pd
import numpy as np

df = pd.read_csv('cleaned_combined.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("Data Loaded Successfully!")
print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")
print(f"\nColumn Names:")
print(list(df.columns))

Data Loaded Successfully!
Rows    : 15368
Columns : 32

Column Names:
['Admin_Unit_Code', 'Plot_Name', 'Location_Type', 'Year', 'Date', 'Start_Time', 'End_Time', 'Observer', 'Visit', 'Interval_Length', 'ID_Method', 'Distance', 'Flyover_Observed', 'Sex', 'Common_Name', 'Scientific_Name', 'AcceptedTSN', 'NPSTaxonCode', 'AOU_Code', 'PIF_Watchlist_Status', 'Regional_Stewardship_Status', 'Temperature', 'Humidity', 'Sky', 'Wind', 'Disturbance', 'Initial_Three_Min_Cnt', 'Month', 'Month_Name', 'Day_Name', 'Week', 'Season']


In [3]:
print("=" * 50)
print("OVERALL SUMMARY")
print("=" * 50)
print(f"Total Observations    : {len(df):,}")
print(f"Unique Species        : {df['Common_Name'].nunique()}")
print(f"National Parks        : {df['Admin_Unit_Code'].nunique()}")
print(f"Unique Observers      : {df['Observer'].nunique()}")
print(f"Date Range            : {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Forest Observations   : {(df['Location_Type']=='Forest').sum():,}")
print(f"Grassland Observations: {(df['Location_Type']=='Grassland').sum():,}")
print(f"Watchlist Sightings   : {df['PIF_Watchlist_Status'].sum():,}")
print(f"Flyover Observed      : {df['Flyover_Observed'].sum():,}")

OVERALL SUMMARY
Total Observations    : 15,368
Unique Species        : 126
National Parks        : 11
Unique Observers      : 3
Date Range            : 2018-05-07 to 2018-07-19
Forest Observations   : 8,542
Grassland Observations: 6,826
Watchlist Sightings   : 378
Flyover Observed      : 689


In [4]:
print("=" * 50)
print("TEMPORAL ANALYSIS")
print("=" * 50)

print("\nMonth-wise Observations:")
monthly = df.groupby(['Month','Month_Name']).size().reset_index(name='Count')
monthly = monthly.sort_values('Month')
print(monthly.to_string(index=False))

print("\nSeason-wise Observations:")
seasonal = df['Season'].value_counts().reset_index()
seasonal.columns = ['Season','Count']
print(seasonal.to_string(index=False))

print("\nDay-wise Observations:")
daily = df['Day_Name'].value_counts().reset_index()
daily.columns = ['Day','Count']
print(daily.to_string(index=False))

TEMPORAL ANALYSIS

Month-wise Observations:
 Month Month_Name  Count
     5        May   4863
     6       June   6209
     7       July   4296

Season-wise Observations:
Season  Count
Summer  10505
Spring   4863

Day-wise Observations:
      Day  Count
  Tuesday   3374
   Monday   3159
 Thursday   2642
Wednesday   2292
 Saturday   1604
   Friday   1557
   Sunday    740


In [5]:
print("=" * 50)
print("HABITAT ANALYSIS")
print("=" * 50)

habitat = df.groupby('Location_Type').agg(
    Total_Observations  = ('Common_Name', 'count'),
    Unique_Species      = ('Common_Name', 'nunique'),
    Watchlist_Sightings = ('PIF_Watchlist_Status', 'sum'),
    Avg_Temperature     = ('Temperature', 'mean'),
    Avg_Humidity        = ('Humidity', 'mean')
).round(2)
print(habitat)

print("\nPark-wise Species Count:")
park = df.groupby('Admin_Unit_Code').agg(
    Observations   = ('Common_Name', 'count'),
    Unique_Species = ('Common_Name', 'nunique')
).sort_values('Unique_Species', ascending=False)
print(park)

HABITAT ANALYSIS
               Total_Observations  Unique_Species  Watchlist_Sightings  \
Location_Type                                                            
Forest                       8542             108                  338   
Grassland                    6826             107                   40   

               Avg_Temperature  Avg_Humidity  
Location_Type                                 
Forest                   21.87         77.76  
Grassland                23.27         69.66  

Park-wise Species Count:
                 Observations  Unique_Species
Admin_Unit_Code                              
MONO                     2522              99
CHOH                     2201              80
ANTI                     3463              80
MANA                     1901              80
NACE                      683              66
PRWI                     2462              54
HAFE                      530              54
GWMP                      385              49
CATO        

In [6]:
print("=" * 50)
print("SPECIES ANALYSIS")
print("=" * 50)

print("\nTop 15 Most Observed Species:")
top15 = df['Common_Name'].value_counts().head(15).reset_index()
top15.columns = ['Species', 'Count']
print(top15.to_string(index=False))

forest_sp = set(df[df['Location_Type']=='Forest']['Common_Name'])
grass_sp  = set(df[df['Location_Type']=='Grassland']['Common_Name'])

print(f"\nSpecies in BOTH habitats : {len(forest_sp & grass_sp)}")
print(f"Species ONLY in Forest   : {len(forest_sp - grass_sp)}")
print(f"Species ONLY in Grassland: {len(grass_sp - forest_sp)}")

print("\nForest Exclusive Species (Top 10):")
only_f = df[df['Common_Name'].isin(forest_sp - grass_sp)]['Common_Name'].value_counts().head(10).reset_index()
only_f.columns = ['Species','Count']
print(only_f.to_string(index=False))

print("\nGrassland Exclusive Species (Top 10):")
only_g = df[df['Common_Name'].isin(grass_sp - forest_sp)]['Common_Name'].value_counts().head(10).reset_index()
only_g.columns = ['Species','Count']
print(only_g.to_string(index=False))

SPECIES ANALYSIS

Top 15 Most Observed Species:
                Species  Count
      Northern Cardinal   1125
          Carolina Wren    993
         Red-eyed Vireo    737
Eastern Tufted Titmouse    720
         Indigo Bunting    611
     Eastern Wood-Pewee    574
          Field Sparrow    492
 Red-bellied Woodpecker    488
         American Robin    470
     Acadian Flycatcher    462
     American Goldfinch    457
       Chipping Sparrow    411
  Blue-gray Gnatcatcher    404
     Carolina Chickadee    365
          Mourning Dove    359

Species in BOTH habitats : 89
Species ONLY in Forest   : 19
Species ONLY in Grassland: 18

Forest Exclusive Species (Top 10):
                    Species  Count
             Hooded Warbler     64
      Louisiana Waterthrush     43
        Worm-eating Warbler     31
Black-throated Blue Warbler      9
           Cerulean Warbler      7
   Double-crested Cormorant      7
                     Osprey      7
    Yellow-throated Warbler      3
              

In [7]:
print("=" * 50)
print("ACTIVITY PATTERNS")
print("=" * 50)

print("\nID Method:")
id_method = df['ID_Method'].value_counts().reset_index()
id_method.columns = ['Method','Count']
id_method['Percentage'] = (id_method['Count']/len(df)*100).round(1)
print(id_method.to_string(index=False))

print("\nInterval Length:")
interval = df['Interval_Length'].value_counts().reset_index()
interval.columns = ['Interval','Count']
interval['Percentage'] = (interval['Count']/len(df)*100).round(1)
print(interval.to_string(index=False))

print("\nDistance from Observer:")
distance = df['Distance'].value_counts().reset_index()
distance.columns = ['Distance','Count']
distance['Percentage'] = (distance['Count']/len(df)*100).round(1)
print(distance.to_string(index=False))

print("\nSex Distribution:")
sex = df['Sex'].value_counts().reset_index()
sex.columns = ['Sex','Count']
sex['Percentage'] = (sex['Count']/len(df)*100).round(1)
print(sex.to_string(index=False))

print("\nFlyover Observed:")
flyover = df['Flyover_Observed'].value_counts().reset_index()
flyover.columns = ['Flyover','Count']
flyover['Percentage'] = (flyover['Count']/len(df)*100).round(1)
print(flyover.to_string(index=False))

ACTIVITY PATTERNS

ID Method:
       Method  Count  Percentage
      Singing   9620        62.6
      Calling   3940        25.6
Visualization   1806        11.8
      Unknown      2         0.0

Interval Length:
    Interval  Count  Percentage
   0-2.5 min   7752        50.4
 2.5 - 5 min   3149        20.5
 5 - 7.5 min   2386        15.5
7.5 - 10 min   2081        13.5

Distance from Observer:
       Distance  Count  Percentage
50 - 100 Meters   7774        50.6
   <= 50 Meters   6905        44.9
        Unknown    689         4.5

Sex Distribution:
         Sex  Count  Percentage
Undetermined  12133        78.9
        Male   3109        20.2
      Female    126         0.8

Flyover Observed:
 Flyover  Count  Percentage
   False  14679        95.5
    True    689         4.5


In [8]:
print("=" * 50)
print("ENVIRONMENTAL ANALYSIS")
print("=" * 50)

print("\nTemperature Stats:")
print(f"  Min    : {df['Temperature'].min():.1f} C")
print(f"  Max    : {df['Temperature'].max():.1f} C")
print(f"  Mean   : {df['Temperature'].mean():.1f} C")
print(f"  Median : {df['Temperature'].median():.1f} C")

print("\nHumidity Stats:")
print(f"  Min    : {df['Humidity'].min():.1f}%")
print(f"  Max    : {df['Humidity'].max():.1f}%")
print(f"  Mean   : {df['Humidity'].mean():.1f}%")
print(f"  Median : {df['Humidity'].median():.1f}%")

print("\nSky Conditions:")
sky = df['Sky'].value_counts().reset_index()
sky.columns = ['Sky','Count']
sky['Percentage'] = (sky['Count']/len(df)*100).round(1)
print(sky.to_string(index=False))

print("\nDisturbance Effect:")
dist = df['Disturbance'].value_counts().reset_index()
dist.columns = ['Disturbance','Count']
dist['Percentage'] = (dist['Count']/len(df)*100).round(1)
print(dist.to_string(index=False))

print("\nWind Conditions:")
wind = df['Wind'].value_counts().reset_index()
wind.columns = ['Wind','Count']
wind['Percentage'] = (wind['Count']/len(df)*100).round(1)
print(wind.to_string(index=False))

ENVIRONMENTAL ANALYSIS

Temperature Stats:
  Min    : 11.0 C
  Max    : 37.3 C
  Mean   : 22.5 C
  Median : 22.2 C

Humidity Stats:
  Min    : 7.3%
  Max    : 98.8%
  Mean   : 74.2%
  Median : 76.6%

Sky Conditions:
                Sky  Count  Percentage
      Partly Cloudy   6172        40.2
Clear or Few Clouds   5330        34.7
    Cloudy/Overcast   2916        19.0
                Fog    598         3.9
       Mist/Drizzle    352         2.3

Disturbance Effect:
             Disturbance  Count  Percentage
      No effect on count   7525        49.0
  Slight effect on count   5836        38.0
Moderate effect on count   1577        10.3
 Serious effect on count    430         2.8

Wind Conditions:
                                      Wind  Count  Percentage
 Light air movement (1-3 mph) smoke drifts   7635        49.7
     Calm (< 1 mph) smoke rises vertically   4210        27.4
  Light breeze (4-7 mph) wind felt on face   3159        20.6
Gentle breeze (8-12 mph), leaves in motion 

In [9]:
print("=" * 50)
print("OBSERVER ANALYSIS")
print("=" * 50)

observer = df.groupby('Observer').agg(
    Observations   = ('Common_Name', 'count'),
    Unique_Species = ('Common_Name', 'nunique'),
    Parks_Covered  = ('Admin_Unit_Code', 'nunique')
).sort_values('Observations', ascending=False)
print(observer)

print("\n" + "=" * 50)
print("CONSERVATION ANALYSIS")
print("=" * 50)

wl = df[df['PIF_Watchlist_Status'] == True]
watchlist = wl.groupby('Common_Name').agg(
    Sightings = ('Common_Name', 'count'),
    Parks     = ('Admin_Unit_Code', 'nunique'),
    Habitat   = ('Location_Type', lambda x: '/'.join(sorted(x.unique())))
).sort_values('Sightings', ascending=False)
print(watchlist)

print(f"\nTotal Watchlist Sightings : {len(wl):,}")
print(f"% of All Observations    : {len(wl)/len(df)*100:.1f}%")

print("\nRegional Stewardship Species Top 10:")
rs = df[df['Regional_Stewardship_Status']==True]['Common_Name'].value_counts().head(10).reset_index()
rs.columns = ['Species','Count']
print(rs.to_string(index=False))

OBSERVER ANALYSIS
                  Observations  Unique_Species  Parks_Covered
Observer                                                     
Elizabeth Oswald          5759             119             11
Kimberly Serno            5346              90             11
Brian Swimelar            4263              83             10

CONSERVATION ANALYSIS
                       Sightings  Parks           Habitat
Common_Name                                              
Wood Thrush                  309     10  Forest/Grassland
Worm-eating Warbler           31      5            Forest
Prairie Warbler               25      3  Forest/Grassland
Cerulean Warbler               7      3            Forest
Willow Flycatcher              2      2         Grassland
Kentucky Warbler               2      2  Forest/Grassland
Blue-winged Warbler            1      1            Forest
Red-headed Woodpecker          1      1            Forest

Total Watchlist Sightings : 378
% of All Observations    : 2.5%

Reg

In [10]:
print("=" * 50)
print("EDA COMPLETE - KEY FINDINGS")
print("=" * 50)
print(f"1. Total {len(df):,} observations across 11 parks")
print(f"2. {df['Common_Name'].nunique()} unique bird species found")
print(f"3. June has highest observations: {df[df['Month']==6]['Common_Name'].count():,}")
print(f"4. Most common species: {df['Common_Name'].value_counts().index[0]}")
print(f"5. Forest species only: {len(set(df[df['Location_Type']=='Forest']['Common_Name']) - set(df[df['Location_Type']=='Grassland']['Common_Name']))}")
print(f"6. Grassland species only: {len(set(df[df['Location_Type']=='Grassland']['Common_Name']) - set(df[df['Location_Type']=='Forest']['Common_Name']))}")
print(f"7. {df['ID_Method'].value_counts().index[0]} is most common ID method: {df['ID_Method'].value_counts().iloc[0]:,}")
print(f"8. Watchlist species sightings: {df['PIF_Watchlist_Status'].sum():,} ({df['PIF_Watchlist_Status'].sum()/len(df)*100:.1f}%)")
print(f"9. Most biodiverse park: {df.groupby('Admin_Unit_Code')['Common_Name'].nunique().idxmax()}")
print(f"10. Female birds only: {(df['Sex']=='Female').sum()} ({(df['Sex']=='Female').sum()/len(df)*100:.1f}%)")

EDA COMPLETE - KEY FINDINGS
1. Total 15,368 observations across 11 parks
2. 126 unique bird species found
3. June has highest observations: 6,209
4. Most common species: Northern Cardinal
5. Forest species only: 19
6. Grassland species only: 18
7. Singing is most common ID method: 9,620
8. Watchlist species sightings: 378 (2.5%)
9. Most biodiverse park: MONO
10. Female birds only: 126 (0.8%)
